# Auto-Landlord — Live-Demo

**Mietportfolio Leopoldstadt (1020 Wien) · Advanced Data Management 2026**

Dieses Notebook klont das Repository und baut die Datenbank aus den
gespeicherten Rohdaten neu auf. Dann zeigt es die Ergebnisse, die in der
Entscheidungsvorlage stehen.

- **Nur lesen, nicht rechnen:** Die Pipeline braucht ausschließlich die
  Python-Standardbibliothek. Pandas und matplotlib dienen hier nur zur
  Anzeige. Jede Zahl stammt aus `data/processed/kennzahl_*.csv`.
- **Mieter:** Die Mieter sind synthetisch, Markt- und KV-Daten sind echt.

**Bedienung:** *Laufzeit → Alle ausführen*, oder Zelle für Zelle mit Shift+Enter.

## 1 · Repository holen und Pipeline laufen lassen

In [ ]:
import os, subprocess
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/TaoWao-dev/AdvancedDataMangement_Project_Auto-Landlord.git", "repo"], check=True)
os.chdir("repo")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# extract -> clean -> Gate 1+2 -> SQLite -> Gate 4 (KV) -> Modell -> Kennzahl-SQL -> Gate 3
lauf = subprocess.run(["python3", "-m", "src.run"], capture_output=True, text=True)
print(lauf.stdout[-3000:] or lauf.stderr[-3000:])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
pd.set_option("display.max_columns", 30)
lies = lambda name: pd.read_csv(f"data/processed/{name}.csv", sep=";")

## 2 · Der Markt: Angebotsmieten je Größenklasse

In [ ]:
markt = lies("kennzahl_marktentwicklung")
markt = markt[markt.im_portfolio == 1]
display(markt[["groessenklasse", "abruf_datum", "fallzahl", "median_eur_m2",
               "belastbarkeit", "veraenderung_seit_erstem_abruf_pct"]])
fig, ax = plt.subplots(figsize=(7, 3.5))
for k, g in markt.groupby("groessenklasse"):
    ax.plot(g.abruf_datum, g.median_eur_m2, marker="o", label=f"{k} m²")
ax.set_ylabel("Median €/m²"); ax.set_title("Angebotsmiete Leopoldstadt"); ax.legend(); plt.show()
print("Achtung: Der erste Abruf (25.09.) war nur die erste Ergebnisseite.")

## 3 · Die KV-Erhöhungen je Branche

Von einem Sprachmodell aus den gespeicherten KV-Seiten gelesen. Gate 4 hat jede Zahl am Wortlaut der Quellzeile geprüft.

In [ ]:
kv = lies("kennzahl_kv_entwicklung")
display(kv.pivot_table(index="branche_id", columns="jahr", values="satz_index").round(2)
          .assign(mieter_im_portfolio=kv.groupby("branche_id").mieter_im_portfolio.first()))

## 4 · Die Empfehlung je Größenklasse

In [ ]:
emp = lies("kennzahl_empfehlung")
display(emp[["groessenklasse", "marktmiete_eur_m2", "vertraege", "erhoehen", "nicht_erhoehen",
             "median_erhoehung_pct", "max_erhoehung_pct", "mittel_p_annahme",
             "bw_empfehlung_summe", "bw_ohne_erhoehung_summe", "mehrwert_gegen_ohne_erhoehung"]])

## 5 · Wer verträgt wie viel? Branche × empfohlene Erhöhung

Jede Zahl in der Tabelle ist die Anzahl der Verträge. Im Diagramm ist jeder Punkt ein Vertrag.

In [ ]:
v = lies("kennzahl_empfehlung_vertrag")
display(pd.crosstab(v.branche_id, v.erhoehung_pct, margins=True, margins_name="Summe"))

fig, ax = plt.subplots(figsize=(8, 4.5))
for b, g in v.groupby("branche_id"):
    ax.scatter(g.luecke_zum_markt_pct, g.erhoehung_pct, label=b, s=60, alpha=0.8)
ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
ax.set_xlabel("Marktmiete über der heutigen Miete (%)")
ax.set_ylabel("empfohlene Erhöhung (%)")
ax.set_title("Erhöht wird vor allem, wo die Miete unter dem Markt liegt"); ax.legend(); plt.show()

## 6 · Ein Vertrag Schritt für Schritt

Im Auswahlfeld einen Vertrag wählen. Für die Präsentation lohnt sich der Vergleich **LS-45** (Handel) mit **LS-29** (Bank).

In [ ]:
def zeige(vertrag):
    out = subprocess.run(["python3", "-m", "src.demo", vertrag], capture_output=True, text=True)
    print(out.stdout or out.stderr)

try:
    import ipywidgets as w
    w.interact(zeige, vertrag=w.Dropdown(options=sorted(v.vertrag_id), value="LS-45", description="Vertrag"))
except ImportError:
    for x in ("LS-45", "LS-29"):
        zeige(x)

## 7 · Woran hängt die Empfehlung? (Sensitivität)

In [ ]:
s = lies("kennzahl_sensitivitaet").sort_values("geaendert", ascending=False)
display(s[["parameter", "rand", "wert", "vertraege", "geaendert", "mittel_erhoehung_pct", "mittel_erhoehung_basis_pct"]])

## 8 · Die Entscheidungsvorlage und das Mieterverzeichnis

In [ ]:
display(Markdown(open("docs/vorlage.md", encoding="utf-8").read()))

In [ ]:
subprocess.run(["python3", "-m", "src.demo"], capture_output=True)   # schreibt docs/demo/mieterverzeichnis.html neu
display(HTML(open("docs/demo/mieterverzeichnis.html", encoding="utf-8").read()))

## 9 · Evals und Tests

Die Evals vergleichen die gespeicherten Modellausgaben mit Referenzen. Das kostet nichts und braucht keine Modellanfrage.

In [ ]:
for cmd in (["python3", "evals/lauf.py"], ["python3", "evals/gegenprobe_grader.py"],
            ["python3", "tests/test_pipeline.py"], ["python3", "tests/test_modell.py"], ["python3", "tests/test_graders.py"]):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("$", " ".join(cmd)); print((r.stdout or r.stderr).strip()[-1500:]); print()